# NepError — Notebook 1: Inference Pipeline
## XLM-R Large & mDeBERTa-v3 on Nepali NLI

---

**Project:** NepError: A Hierarchical Diagnostic Taxonomy of Failure Modes in Nepali NLI  
**Author:** Pema Tshering Sherpa, RV University Bengaluru  
**Collaborator:** Prof. Bal Krishna Bal, KU ILP Lab, Kathmandu University  

---

### What this notebook does

This notebook runs inference on the Nepali NLI datasets using two pre-trained multilingual models that require **no fine-tuning** — they already have NLI classification heads trained on large multilingual NLI corpora:

| Model | HuggingFace ID | NLI Training Data |
|-------|---------------|-------------------|
| XLM-R Large | `joeddav/xlm-roberta-large-xnli` | XNLI (15 languages, ~400K pairs) |
| mDeBERTa-v3 Base | `MoritzLaurer/mDeBERTa-v3-base-xnli-multilingual-nli-2mil7` | 2.7M multilingual NLI pairs |

For each model, this notebook will:
1. Load the IRIIS-RESEARCH Nepali NLI datasets from HuggingFace
2. Run full inference on the test splits
3. Collect every **wrong prediction** with confidence scores
4. Save a clean, annotation-ready CSV of failure cases

### Output files
```
results/
  failures_xlmr_mnli.csv
  failures_xlmr_rte.csv
  failures_mdeberta_mnli.csv
  failures_mdeberta_rte.csv
  inference_summary.csv        ← accuracy, failure counts, per-label breakdown
```

> **Note on label encoding:**  
> MNLI/XNLI: `0=entailment`, `1=neutral`, `2=contradiction`  
> RTE: `0=entailment`, `1=not_entailment`  
> QNLI: `0=entails` (question answered), `1=not_entails`

---
## Section 0 — Setup & Installation

In [ ]:
# Install required packages
# Run this cell once at the start of your Colab session
!pip install -q transformers datasets torch pandas numpy scikit-learn tqdm

In [ ]:
import os
import json
import warnings
from pathlib import Path
from dataclasses import dataclass, field
from typing import Dict, List, Tuple, Optional

import numpy as np
import pandas as pd
import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from torch.utils.data import DataLoader, Dataset as TorchDataset
from sklearn.metrics import classification_report, confusion_matrix
from tqdm.auto import tqdm

warnings.filterwarnings('ignore')

# Verify GPU availability
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# ─────────────────────────────────────────────
#  CONFIGURATION — edit these if needed
# ─────────────────────────────────────────────

@dataclass
class InferenceConfig:
    # Batch size — reduce to 8 if you get CUDA out-of-memory on XLM-R Large
    batch_size: int = 16

    # Max token length — Nepali sentences are generally short, 128 is sufficient
    max_length: int = 128

    # Output directory for results
    output_dir: str = "results"

    # Models to evaluate
    models: Dict[str, str] = field(default_factory=lambda: {
        "xlmr":     "joeddav/xlm-roberta-large-xnli",
        "mdeberta": "MoritzLaurer/mDeBERTa-v3-base-xnli-multilingual-nli-2mil7",
    })

    # Datasets to evaluate — (hf_dataset_id, text_col_1, text_col_2, label_col, task_type)
    # task_type: 'nli3' = 3-class NLI, 'nli2' = binary NLI
    datasets: List[Tuple] = field(default_factory=lambda: [
        ("IRIIS-RESEARCH/MNLI-Nepali",  "premise",   "hypothesis", "label", "nli3"),
        ("IRIIS-RESEARCH/RTE-Nepali",   "sentence1", "sentence2",  "label", "nli2"),
    ])

cfg = InferenceConfig()
Path(cfg.output_dir).mkdir(exist_ok=True)
print("Config loaded. Output will be saved to:", cfg.output_dir)

---
## Section 1 — Load Datasets

In [ ]:
def load_nli_dataset(hf_id: str, text_col1: str, text_col2: str,
                     label_col: str) -> pd.DataFrame:
    """
    Load a HuggingFace NLI dataset and return the test split as a DataFrame.
    Normalises column names to: premise, hypothesis, label.
    """
    print(f"  Loading {hf_id} ...")
    ds = load_dataset(hf_id)
    df = ds['test'].to_pandas()

    # Normalise column names so all downstream code is consistent
    df = df.rename(columns={
        text_col1: 'premise',
        text_col2: 'hypothesis',
        label_col: 'label',
    })

    # Cast label to int (QNLI uses float64)
    df['label'] = df['label'].astype(int)

    # Add a stable row ID for annotation tracking
    df.insert(0, 'instance_id', [f"{hf_id.split('/')[-1]}_test_{i}" for i in range(len(df))])

    print(f"    Rows: {len(df):,}  |  Label distribution: {df['label'].value_counts().to_dict()}")
    return df


print("Loading test splits...")
loaded_datasets = {}
for (hf_id, col1, col2, label_col, task_type) in cfg.datasets:
    name = hf_id.split('/')[-1].replace('-Nepali', '').lower()
    loaded_datasets[name] = {
        'df': load_nli_dataset(hf_id, col1, col2, label_col),
        'task_type': task_type,
        'hf_id': hf_id,
    }

print("\nAll datasets loaded.")

---
## Section 2 — Inference Engine

In [ ]:
class NLIPairDataset(TorchDataset):
    """
    PyTorch Dataset that tokenises premise-hypothesis pairs.
    Handles the [CLS] premise [SEP] hypothesis [SEP] format
    expected by all BERT-family NLI models.
    """
    def __init__(self, premises: List[str], hypotheses: List[str],
                 tokenizer, max_length: int):
        self.premises    = premises
        self.hypotheses  = hypotheses
        self.tokenizer   = tokenizer
        self.max_length  = max_length

    def __len__(self):
        return len(self.premises)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.premises[idx],
            self.hypotheses[idx],
            max_length=self.max_length,
            padding='max_length',
            truncation=True,
            return_tensors='pt',
        )
        return {
            'input_ids':      encoding['input_ids'].squeeze(0),
            'attention_mask': encoding['attention_mask'].squeeze(0),
        }

In [ ]:
def run_inference(model_name: str, model_hf_id: str,
                  dataset_name: str, df: pd.DataFrame,
                  task_type: str) -> pd.DataFrame:
    """
    Run full inference for one model on one dataset.

    Returns a DataFrame of ALL predictions (not just failures).
    Failure filtering happens in the next section.

    Columns returned:
        instance_id, premise, hypothesis, true_label,
        pred_label, correct,
        conf_label0, conf_label1, conf_label2 (or conf_label0/1 for binary),
        model, dataset
    """
    print(f"\n{'='*60}")
    print(f"  Model:   {model_name}")
    print(f"  Dataset: {dataset_name}  ({len(df):,} rows)")
    print(f"{'='*60}")

    # ── Load model and tokenizer ──────────────────────────────
    print("  Loading tokenizer and model...")
    tokenizer = AutoTokenizer.from_pretrained(model_hf_id)
    model     = AutoModelForSequenceClassification.from_pretrained(model_hf_id)
    model.to(DEVICE)
    model.eval()

    num_labels = model.config.num_labels
    print(f"  Model loaded. Num labels: {num_labels}")

    # ── Build DataLoader ──────────────────────────────────────
    torch_dataset = NLIPairDataset(
        premises=df['premise'].tolist(),
        hypotheses=df['hypothesis'].tolist(),
        tokenizer=tokenizer,
        max_length=cfg.max_length,
    )
    loader = DataLoader(torch_dataset, batch_size=cfg.batch_size,
                        shuffle=False, num_workers=2, pin_memory=True)

    # ── Run inference ─────────────────────────────────────────
    all_probs = []
    with torch.no_grad():
        for batch in tqdm(loader, desc=f"  Inference"):
            input_ids      = batch['input_ids'].to(DEVICE)
            attention_mask = batch['attention_mask'].to(DEVICE)
            logits = model(input_ids=input_ids,
                           attention_mask=attention_mask).logits
            probs  = torch.softmax(logits, dim=-1).cpu().numpy()
            all_probs.append(probs)

    all_probs = np.vstack(all_probs)          # shape: (N, num_labels)
    pred_labels = np.argmax(all_probs, axis=1)

    # ── Map model's internal label IDs to MNLI convention ────
    # Some models order labels differently (e.g. entailment=2 internally).
    # We remap using model.config.id2label to a canonical mapping.
    id2label = model.config.id2label
    print(f"  Model label mapping: {id2label}")

    canonical = {'entailment': 0, 'neutral': 1, 'contradiction': 2,
                 'not_entailment': 1}

    # Build a remapping array: model_internal_id → canonical_id
    remap = np.arange(num_labels)
    for internal_id, label_str in id2label.items():
        label_key = label_str.lower().replace(' ', '_')
        if label_key in canonical:
            remap[int(internal_id)] = canonical[label_key]

    pred_labels_canonical = np.array([remap[p] for p in pred_labels])

    # Also reorder probability columns to canonical order
    conf_cols = {}
    for internal_id, label_str in id2label.items():
        label_key = label_str.lower().replace(' ', '_')
        canonical_id = canonical.get(label_key, int(internal_id))
        conf_cols[f'conf_label{canonical_id}'] = all_probs[:, int(internal_id)]

    # ── Assemble results DataFrame ────────────────────────────
    results = df[['instance_id', 'premise', 'hypothesis', 'label']].copy()
    results = results.rename(columns={'label': 'true_label'})
    results['pred_label'] = pred_labels_canonical
    results['correct']    = (results['true_label'] == results['pred_label']).astype(int)
    results['model']      = model_name
    results['dataset']    = dataset_name

    for col_name, col_vals in sorted(conf_cols.items()):
        results[col_name] = np.round(col_vals, 4)

    # ── Print quick diagnostics ───────────────────────────────
    accuracy = results['correct'].mean()
    n_errors = (results['correct'] == 0).sum()
    print(f"\n  Accuracy : {accuracy:.4f} ({accuracy*100:.2f}%)")
    print(f"  Failures : {n_errors:,} / {len(results):,}")
    print(f"\n  Classification Report:")

    # Use the actual unique predicted/true labels to avoid shape mismatch.
    # XLM-R always has 3 output nodes even when the dataset is binary (RTE).
    # We pass labels= explicitly so sklearn only reports on classes that exist.
    unique_labels = sorted(set(results['true_label'].tolist() + results['pred_label'].tolist()))
    full_label_names = ['entailment', 'neutral', 'contradiction']
    target_names = [full_label_names[i] for i in unique_labels if i < len(full_label_names)]
    print(classification_report(
        results['true_label'], results['pred_label'],
        labels=unique_labels, target_names=target_names, zero_division=0
    ))

    # Free GPU memory before loading next model
    del model
    torch.cuda.empty_cache()

    return results

---
## Section 3 — Run All Models × All Datasets

In [ ]:
# This cell runs inference for every (model, dataset) combination.
# On a Colab T4 GPU, expect roughly:
#   - XLM-R Large on MNLI (10,200 rows): ~8 minutes
#   - mDeBERTa Base on MNLI (10,200 rows): ~4 minutes
#   - RTE runs are fast (503 rows each): < 1 minute each

all_results = {}  # key: "model_dataset", value: full results DataFrame

for model_name, model_hf_id in cfg.models.items():
    for dataset_name, dataset_info in loaded_datasets.items():
        run_key = f"{model_name}_{dataset_name}"
        results_df = run_inference(
            model_name=model_name,
            model_hf_id=model_hf_id,
            dataset_name=dataset_name,
            df=dataset_info['df'],
            task_type=dataset_info['task_type'],
        )
        all_results[run_key] = results_df

print("\n" + "="*60)
print("All inference runs complete.")
print("="*60)

---
## Section 4 — Extract & Save Failure Cases

In [ ]:
def extract_failures(results_df: pd.DataFrame,
                     max_per_class: int = 200) -> pd.DataFrame:
    """
    Extract failure cases from a results DataFrame.

    Applies stratified sampling by true_label so annotation batches
    are balanced across entailment/neutral/contradiction errors.

    Also adds automated pre-filter flags to help annotators:
      flag_romanization   — premise or hypothesis is predominantly Latin script (T1b candidate)
      flag_codeswitching  — Devanagari + Latin mixed in same sentence (T3a candidate)
      flag_negation       — contains known Nepali negation markers (T2a candidate)
    """
    failures = results_df[results_df['correct'] == 0].copy()

    # Stratified sample: equal failure cases per true label
    label_counts = failures['true_label'].value_counts()
    n_per_class  = min(max_per_class, label_counts.min())

    sampled = (
        failures
        .groupby('true_label', group_keys=False)
        .apply(lambda g: g.sample(n=n_per_class, random_state=42))
        .reset_index(drop=True)
    )

    # ── Automated pre-filter flags ────────────────────────────
    # These are hints for annotators, not final labels.
    # Annotators should still verify each flag manually.

    import re

    def is_predominantly_latin(text: str, threshold: float = 0.5) -> bool:
        """True if more than `threshold` fraction of characters are ASCII letters."""
        letters = re.findall(r'[a-zA-Z\u0900-\u097F]', text)
        if not letters:
            return False
        ascii_count = sum(1 for c in letters if c.isascii())
        return (ascii_count / len(letters)) > threshold

    def has_code_switching(text: str) -> bool:
        """True if text contains both Devanagari and Latin script words."""
        has_dev   = bool(re.search(r'[\u0900-\u097F]', text))
        has_latin = bool(re.search(r'[a-zA-Z]{2,}', text))
        return has_dev and has_latin

    NEGATION_MARKERS = [
        'छैन', 'छैनन्', 'दैन', 'दैनन्', 'होइन', 'होइनन्',
        'नाई', 'नगर्', 'नहुने', 'नभएको', 'नगरेको'
    ]
    def has_negation(text: str) -> bool:
        return any(marker in text for marker in NEGATION_MARKERS)

    combined_text = sampled['premise'] + ' ' + sampled['hypothesis']

    sampled['flag_romanization']  = combined_text.apply(is_predominantly_latin)
    sampled['flag_codeswitching'] = combined_text.apply(has_code_switching)
    sampled['flag_negation']      = combined_text.apply(has_negation)

    # Blank annotation columns for the human annotator to fill in
    sampled['taxonomy_tier']  = ''   # T1a / T1b / T2a / T2b / T3a / T3b / T4a / T4b
    sampled['annotator_note'] = ''
    sampled['annotator_id']   = ''

    return sampled


summary_rows = []

for run_key, results_df in all_results.items():
    model_name, dataset_name = run_key.split('_', 1)

    # Extract failures
    failures = extract_failures(results_df, max_per_class=200)

    # Save failures CSV
    out_path = f"{cfg.output_dir}/failures_{run_key}.csv"
    failures.to_csv(out_path, index=False, encoding='utf-8-sig')
    print(f"Saved {len(failures):,} failure cases → {out_path}")

    # Collect summary stats
    n_total    = len(results_df)
    n_correct  = results_df['correct'].sum()
    n_failures = n_total - n_correct
    accuracy   = n_correct / n_total

    flags = failures[['flag_romanization', 'flag_codeswitching', 'flag_negation']]

    summary_rows.append({
        'model':           model_name,
        'dataset':         dataset_name,
        'n_total':         n_total,
        'n_correct':       n_correct,
        'n_failures':      n_failures,
        'accuracy':        round(accuracy, 4),
        'failures_sampled': len(failures),
        'flag_romanization_count':  flags['flag_romanization'].sum(),
        'flag_codeswitching_count': flags['flag_codeswitching'].sum(),
        'flag_negation_count':      flags['flag_negation'].sum(),
    })

# Save summary table
summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(f"{cfg.output_dir}/inference_summary.csv", index=False)
print(f"\nSummary saved → {cfg.output_dir}/inference_summary.csv")

---
## Section 5 — Summary Table & Visualisation

In [ ]:
print("\n" + "="*60)
print("INFERENCE SUMMARY")
print("="*60)
display_cols = ['model', 'dataset', 'n_total', 'n_failures', 'accuracy',
                'flag_negation_count', 'flag_codeswitching_count', 'flag_romanization_count']
print(summary_df[display_cols].to_string(index=False))

In [ ]:
# ── Inspect a sample of failure cases ────────────────────────
# Change this to look at different model/dataset combinations
INSPECT_RUN = 'xlmr_mnli'

failures_sample = all_results[INSPECT_RUN][all_results[INSPECT_RUN]['correct'] == 0]
failures_sample = failures_sample.head(10)

LABEL_MAP = {0: 'entailment', 1: 'neutral', 2: 'contradiction'}

print(f"\nSample failures from: {INSPECT_RUN}")
print("-" * 60)
for _, row in failures_sample.iterrows():
    print(f"ID:         {row['instance_id']}")
    print(f"Premise:    {row['premise'][:120]}")
    print(f"Hypothesis: {row['hypothesis'][:120]}")
    print(f"True:       {LABEL_MAP.get(row['true_label'], row['true_label'])}  "
          f"→  Predicted: {LABEL_MAP.get(row['pred_label'], row['pred_label'])}")
    print("-" * 60)

---
## Next Steps

After running this notebook you should have:
- `results/failures_xlmr_mnli.csv` — ready for annotation
- `results/failures_mdeberta_mnli.csv` — ready for annotation  
- `results/inference_summary.csv` — accuracy comparison table

**Proceed to:**
- `02_chipsal_bert_finetune.ipynb` — fine-tune CHiPSAL-BERT on MNLI-Nepali, then re-run inference to generate `failures_chipsalbert_mnli.csv`
- Begin manual annotation using the taxonomy guidelines document, opening the failures CSVs in Label Studio or directly in Excel/Google Sheets